# Vajra — 03: Stage 1 Spherical GNN Training & Tracking
### MoES / NCMRWF Problem Statement 26078
This notebook demonstrates building the spherical icosahedral mesh (Level 5 ~50km spacing) and training the GATv2 + GRU spatio-temporal message passing graph neural network to track anomalies.

In [ ]:
import torch
from vajra.tracking.mesh import SphericalMeshBuilder
from vajra.tracking.gnn_model import WeatherGNN
from vajra.tracking.anomaly_tracker import SpatioTemporalTracker

# Build Icosahedral Mesh (Level 2 for rapid demonstration)
mesh_builder = SphericalMeshBuilder(subdivision_level=2)
graph = mesh_builder.build_graph()
print(f"Mesh nodes: {graph.num_nodes}, Mesh edges: {graph.num_edges}")

In [ ]:
# Instantiate Spatio-Temporal Weather GNN
model = WeatherGNN(in_channels=8, hidden_dim=64, heads=4)

# Forward pass on temporal sequence [T=5 lead times, N nodes, C=8 atmospheric features]
x_seq = torch.randn(5, graph.num_nodes, 8)
edge_attr = torch.randn(graph.num_edges, 3)

outputs = model(x_seq, graph.edge_index, edge_attr)
print("Event Probability (sample):", outputs["event_probability"][:5].flatten())
print("Severity Index (sample):", outputs["severity"][:5].flatten())
print("Centroid Delta (sample):", outputs["centroid_delta"][:5])

In [ ]:
# 50-Member Ensemble Trajectory & Uncertainty Cone
tracker = SpatioTemporalTracker(prob_threshold=0.7)
member_trajectories = [
    [[13.5 + i*0.02, 87.2 + i*0.03], [16.8 + i*0.04, 87.0 + i*0.05], [19.8 + i*0.06, 86.8 + i*0.07]]
    for i in range(20)
]
cone_data = tracker.compute_ensemble_uncertainty_cone(member_trajectories)
print("Consensus Trajectory:", cone_data["consensus_trajectory"])
print("90% Cone Geometry Type:", cone_data["uncertainty_cone_90pct"]["type"])